# Task 5: Fine-tune BERT for POS Tagging and Chunking

## Install Required Libraries

In [ ]:
!pip install transformers datasets seqeval
!pip install datasets==2.19.0

 Libraries installed successfully

## Task 1: Dataset Selection

In [ ]:
from datasets import load_dataset

dataset = load_dataset('conll2003')
print(dataset)

 DatasetDict with train, validation, test splits

## Labels

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

print("POS Labels:", label_list)
print("Chunk Labels:", chunk_labels)

 List of POS and Chunk labels

## Task 2: Tokenization

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

 Tokenizer loaded

## Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples, label_type="pos_tags"):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples[label_type]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## Apply Preprocessing

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True
)

 Tokenized dataset with labels aligned

## Task 3: Model Setup

In [ ]:
import torch
from transformers import AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list)
)

model.to(device)

 Model loaded on CPU/GPU

## Task 4: Training

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1
)

 Training arguments configured

## Evaluation Metrics

In [ ]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

## Trainer Setup

In [ ]:
from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## Train Model

In [ ]:
trainer.train()

 Training progress bar and loss decreasing

## Task 5: Evaluation

In [ ]:
trainer.evaluate()

 Precision, Recall, F1 Score values

## Task 6: Inference

In [ ]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=-1)
    predicted_labels = [label_list[p.item()] for p in predictions[0]]

    for token, label in zip(tokens, predicted_labels):
        print(f"{token} -> {label}")

predict("John works at Google in California")

 Each word mapped to POS tag


## Summary

This project demonstrates how BERT can be fine-tuned for token classification tasks like POS Tagging and Chunking. 
The model learns contextual representations and predicts labels for each token effectively.

## Differences

- POS Tagging: Word-level classification (Noun, Verb, Adjective)
- Chunking: Phrase-level grouping (NP, VP)
- POS is simpler, while Chunking is more complex

## Challenges

- Subword token alignment
- Handling -100 labels
- Limited training data may reduce accuracy
- Longer training time on CPU

## Objective

To build a transformer-based model using BERT that can perform:
- POS Tagging for grammatical understanding
- Chunking for phrase detection
